In [1]:
import ast
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
from tqdm import tqdm

datasets = Path("/nas/cee-water/cjgleason/data")
era5_dir = datasets / "ERA5-Land/sub_basin_timeseries"
swot_lake_dir = datasets / 'hydrocron' / 'lake'

save_dir = Path("/nas/cee-water/cjgleason/ted/swot-ml/data/reservoirs")
preprocess_dir = save_dir / "preprocess"
metadata_dir = save_dir / "metadata"

zarr_path = '/scratch3/workspace/tlanghorst_umass_edu-swot-ml-zarr'

matchups = gpd.read_parquet(metadata_dir / "All_MERIT_matchups.parquet").set_index('comid')
matchups.index = matchups.index.astype(str)

In [2]:
from data import BasinDataLake

root_dir = save_dir / 'datalakes' / 'training'
store = BasinDataLake(root_dir)

In [2]:
import global_gauges as gg

facade = gg.GaugeDataFacade()
# sites = facade.get_stations_n_days(90)

In [3]:
sites = facade.get_stations_n_days(90)

/nas/cee-water/cjgleason/ted/swot-ml/.venv/lib/python3.11/site-packages/global_gauges/facade.py:192: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(provider_info)


In [6]:
sites

,name,area,active,latitude,longitude,last_updated,min_date,max_date,min_discharge,max_discharge,mean_discharge,count_discharge,provider_misc,geometry,provider
site_id,,,,,,,,,,,,,,,
UKEA-037048U,Ulting Sarasota,NaN,True,51.746683,0.624437,2025-09-09,2008-10-31,2025-09-07,0.006000,53.471000,2.915773,5063.0,{'guid': '052d0819-2a32-47df-9b99-c243c9c8235b'},POINT (0.62444 51.74668),ukea
UKEA-510810,Beggearn Huish,NaN,True,51.146322,-3.373695,2025-09-09,1967-01-01,2025-09-07,0.026000,11.051000,0.892094,20432.0,{'guid': '48513a18-e485-4317-ae92-93bf4f7f3e54'},POINT (-3.3737 51.14632),ukea
UKEA-F0803,Adwick,NaN,True,53.512713,-1.282505,2025-09-09,1976-01-05,2025-09-07,0.513000,72.728000,3.536529,18081.0,{'guid': 'f22f80f8-1bb0-4e77-b225-291487060c6f'},POINT (-1.2825 53.51271),ukea
UKEA-SS90F011,Thorverton,NaN,True,50.804172,-3.511302,2025-09-09,1956-04-30,2025-09-07,0.421000,286.133000,16.177839,25328.0,{'guid': '3c4d4f78-2d0e-474a-b884-65a9daca18fb'},POINT (-3.5113 50.80417),ukea
UKEA-521410,Iwood,NaN,True,51.363977,-2.788890,2025-09-09,1973-03-13,2025-09-07,0.096000,16.227000,0.813153,18047.0,{'guid': '959f3e4f-bb6e-4f4a-8082-0157eea99482'},POINT (-2.78889 51.36398),ukea
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
USGS-04010500,PIGEON RIVER AT MIDDLE FALLS NR GRAND PORTAGE MN,1577.294918,True,48.012222,-89.616204,2025-09-09,1950-01-01,2025-09-08,0.028317,339.802159,14.054779,27644.0,None,POINT (-89.6162 48.01222),usgs
USGS-04015330,"KNIFE RIVER NEAR TWO HARBORS, MN",216.521930,True,46.946940,-91.795577,2025-09-09,1974-07-01,2025-09-08,0.000283,336.970474,2.560661,18525.0,None,POINT (-91.79558 46.94694),usgs
USGS-04015438,"ST. LOUIS RIVER NEAR SKIBO, MN",261.587499,True,47.481111,-92.040000,2025-09-09,2011-08-16,2025-09-07,0.005380,31.431700,2.357346,5135.0,None,POINT (-92.04 47.48111),usgs


In [7]:
sites.loc['ABOM-134767010']

name               MACLEAY RIVER AT TURNERS FLAT
area                                         NaN
active                                     False
latitude                              -31.007617
longitude                             152.712496
last_updated                 2025-09-09 00:00:00
min_date                     1970-01-16 00:00:00
max_date                     2025-09-06 00:00:00
min_discharge                              0.001
max_discharge                           7331.047
mean_discharge                         48.906446
count_discharge                          20042.0
provider_misc                               None
geometry           POINT (152.712496 -31.007617)
provider                                    abom
Name: ABOM-134767010, dtype: object

In [4]:
# processing_status = store.get_processing_status(source='gauge')
# processed_basins = processing_status.index.get_level_values('subbasin')
# to_process = matchups[~matchups.index.isin(processed_basins)]
to_process = matchups

max_batch_size = 64
for basin_id, basin_group in tqdm(to_process.groupby('outlet_id')):
    subbasin_data = {}
    for comid, row in basin_group.iterrows():
        if not row['custom']:
            gauge_df = None
        else:
            # There is not an explicit flag for custom that is not a gauge, so we just try
            # and then catch the error if this comid doesn't exist in our gauge dataset.
            try:
                gauge_df = facade.get_daily_values(comid).droplevel(axis=0, level=0)
                gauge_df = gauge_df[['discharge']]
                gauge_df['sp_discharge'] = gauge_df / row['total_area']
            except ValueError:
                gauge_df = None
                
        subbasin_data[comid] = gauge_df

        if len(subbasin_data)==max_batch_size:
            store.write_dynamic(basin_id, 'gauge', subbasin_data, mode='append')
            subbasin_data = {}
            
    # Write any remaining data.
    if len(subbasin_data)>0:
        store.write_dynamic(basin_id, 'gauge', subbasin_data, mode='append')

100%|██████████| 643/643 [3:00:58<00:00, 16.89s/it]    


In [ ]:
next(generate_tasks(matchups))

In [ ]:
gauge_df

In [ ]:
df = store.read_dynamic(basin_id, comid, 'gauge', '2020-01-01', '2020-12-31')['gauge']

In [ ]:
subbasin_data[comid]

In [ ]:
gauge_df['sp_discharge'].max()

In [ ]:
processed_basins = store.get_processing_status(source='swot-river')
processed_basins

In [ ]:
processed_basins[processed_basins['has_data']]['basin'].value_counts()

In [ ]:
store.compact()

In [ ]:
matchups['lake_pld_ids'].apply(len).gt(0).sum()